## HW 4
### Choosing our Dataset
We are classifying mushrooms -- poisonous or not! Our data set has 22 features for 8124 entries, with the first feature being our targeted 'is it poisonous' feature.

In [64]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd;
  
# fetch dataset 
mushroom = fetch_ucirepo(id=73) 
  
# data (as pandas dataframes) 
X = mushroom.data.features 
y = mushroom.data.targets 
  
# metadata 
print(mushroom.metadata) 
  
# variable information 
print(mushroom.variables) 


{'uci_id': 73, 'name': 'Mushroom', 'repository_url': 'https://archive.ics.uci.edu/dataset/73/mushroom', 'data_url': 'https://archive.ics.uci.edu/static/public/73/data.csv', 'abstract': 'From Audobon Society Field Guide; mushrooms described in terms of physical characteristics; classification: poisonous or edible', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 8124, 'num_features': 22, 'feature_types': ['Categorical'], 'demographics': [], 'target_col': ['poisonous'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1981, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5959T', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': "This data set includes descriptions of hypothetical samples corresponding to 23 species of gilled mushrooms in the Agaricus and Lepiota Family (pp. 500-525).  Each species is identified as definitely edible, definitely po

### Pre-processing the data
Feature 11 is missing close to 1/3rd of its values. It would be reasonable to drop the feature because it is so incomplete, but I will instead choose to preserve it, replacing missing values with the 'missing' category, in case there is a meaningful signal in that data.

Then we one-hot encode our 22 categorical features into 95 true/false features. This is standard but I'll explain it for my own benefit and the homework's: we used one-hot encoding, which 'drops' categorizations that are now implicitly present in the data set (e.g. when cap-shape is not conical, convex, or any other represented shape it is the dropped cap-shape 'bell').

For KNN and SVM scaling data is standard but all of our data is binary, thus already on the same scale.

In [65]:
X["stalk-root"] = X["stalk-root"].fillna('missing')
X_encoded = pd.get_dummies(X, drop_first = True)

y_encoded = y['poisonous'].map({'e': 0, 'p': 1}) #Edible to 0, Poisonous to 1

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded,
    test_size=0.2, #80/20
    random_state=13, #setting random seed for reproducibility
    stratify=y #match poisonous/non-poisonous distribution across train and test
)

### Training and evaluating our models
##### Logistic Regression

In [84]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

logreg = LogisticRegression(max_iter=1000, random_state=13)
logreg.fit(X_train, y_train)

y_pred_logreg = logreg.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_logreg))
print("Precision:", precision_score(y_test, y_pred_logreg))
print("Recall:", recall_score(y_test, y_pred_logreg))
print("F1:", f1_score(y_test, y_pred_logreg))

# or get all of them at once, broken out by class
print(classification_report(y_test, y_pred_logreg, target_names=['edible', 'poisonous']))

#Sanity checking because we got perfect predictions...
# print('poisonous' in X_encoded.columns) #Did we accidetally leave our target in X?
# print(X_encoded.columns)

print(confusion_matrix(y_test, y_pred_logreg))

coef_df = pd.DataFrame({
    'feature': X_encoded.columns,
    'coefficient': logreg.coef_[0],
    'count_true': X_encoded.sum().values #added to determine how prevalent high-coefficient features are in the dataset
})

coef_df['abs_coef'] = coef_df['coefficient'].abs()
coef_df_sorted = coef_df.sort_values('abs_coef', ascending=False)

print(coef_df_sorted.head(60))

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
              precision    recall  f1-score   support

      edible       1.00      1.00      1.00       842
   poisonous       1.00      1.00      1.00       783

    accuracy                           1.00      1625
   macro avg       1.00      1.00      1.00      1625
weighted avg       1.00      1.00      1.00      1625

[[842   0]
 [  0 783]]
                       feature  coefficient  count_true  abs_coef
80         spore-print-color_r     4.714185          72  4.714185
18                      odor_c     3.959306         192  3.959306
22                      odor_n    -3.922474        3528  3.922474
28                 gill-size_n     3.484096        2512  3.484096
19                      odor_f     3.467432        2160  3.467432
23                      odor_p     2.743404         256  2.743404
27              gill-spacing_w    -2.575962        1312  2.575962
78         spore-print-color_n    -2.187503        1968  2.187503
41     

*Evaluation*:
The biggest surprise here is that this model is perfectly categorizing our training data. With this much data and a task that boils down to distinguishing the species of a fungus, it is within the realm of possibility that perfect classification is possible. 

How is logistic regression making these distinctions? 
Our top coefficient was that mushrooms with a black spore print were poisonous, but this was a rare predictor covering only 72 entries in our dataset.

Disregarding that predictor, our two strongest predictors were *odor* and *gill-size*. Nearly half of our data-set had no odor and was not poisonous. Another 2160 entries in our sample that smelled 'foul' were, unsurprisingly, strongly correlated with a 'poisonous' classficiation. Another three odors across 1024 samples ('creosote', 'pungent', and 'spicy') correlated with poisonous classification, and 400 entries with the 'anise' odor strongly correlated with edibility. 'Narrow' gill-size, as opposed to 'broad', strongly correlated with poisonous mushrooms. 

These features did not dominate our analysis: over half our features had a meaningful signaling effect, with coefficents not dipping below 10% of 'odor_c's 3.96 coefficient until the 58th sorted entry (gill-attachment_f at a coefficient of 0.45).

##### KNN

What k should we be starting with? Our maximum meaningful k would the number of species that we are classifying, and our minimum would be the minimum features needed to identify whether a fungus is poisonous.

In [88]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

k_values = [5, 15, 50, 80]
knn_results = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    
    knn_results.append({
        'k': k,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred)
    })

knn_results_df = pd.DataFrame(knn_results)
print(knn_results_df)

    k  accuracy  precision    recall        f1
0   5  1.000000   1.000000  1.000000  1.000000
1  15  0.998769   1.000000  0.997446  0.998721
2  50  0.998154   0.998721  0.997446  0.998083
3  80  0.992615   0.987358  0.997446  0.992376
